In [29]:
pip install pulp

Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd

food_candidates = pd.read_csv("./final_data_frame.csv")


In [31]:
patient_target = {
    "diet_type": "DM_HT_OBESITY",
    "energy_kcal": 1600,
    "carb_g": 240,
    "protein_g": 60,
    "fat_g": 44,
    "sodium_mg_max": 2000,
    "fiber_g_min": 25
}

In [32]:
def filter_foods_for_disease(df, diet_type):
    filtered = df.copy()

    # DM: remove sugar category if exists
    if "DM" in diet_type:
        filtered = filtered[filtered["category_code"] != "G"]

    # Hypertension: remove very high sodium foods
    if "HT" in diet_type:
        filtered = filtered[
            filtered["sodium_mg_per_portion"].isna() |
            (filtered["sodium_mg_per_portion"] <= 400)
        ]

    # Obesity: remove very high energy per portion
    if "OBESITY" in diet_type:
        filtered = filtered[
            filtered["energy_kcal_per_portion"] <= 250
        ]

    return filtered


candidate_for_patient = filter_foods_for_disease(
    food_candidates,
    patient_target["diet_type"]
)

print("Candidate foods after disease filter:", len(candidate_for_patient))

display(
    candidate_for_patient[[
        "food_code",
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]].head(20)
)

Candidate foods after disease filter: 211


,food_code,food_name,category_code,urt,gram_per_portion,energy_kcal_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
5,AP004,"Tepung beras, mentah",MP,8 Sendok Makan,50.0,176.50,40.000,2.50,1.200
6,AP006,"Bihun, mentah",MP,1/2 Gelas,50.0,174.00,41.050,2.50,0.600
7,AP020,"Makaroni, mentah",MP,1½ Gelas,50.0,176.50,39.350,2.50,2.450
8,AP024,Roti putih,MP,3 Iris,70.0,173.60,35.000,63.70,6.370
9,AP025,Tepung terigu,MP,5 Sendok Makan,50.0,166.50,38.600,1.00,0.150
10,AP029,Biskuit,MP,4 Buah Besar,40.0,183.20,30.040,8.12,0.840
11,BR013,"Kentang, segar",MP,2 Buah Sedang,210.0,130.20,28.350,14.70,1.050
12,BR014,"Kentang hitam, segar",MP,12 Biji,125.0,177.50,42.125,27.50,6.750
13,CR018,"Kacang kedelai, segar",LN,2 Sendok Makan,25.0,71.50,7.525,7.00,0.725
14,CR027,"Kacang merah, segar",LN,2 Sendok Makan,25.0,42.75,7.000,1.75,0.525


In [33]:
import pandas as pd
import numpy as np
import pulp

# ============================================================
# 1. Prepare MILP input data
# ============================================================

milp_df = candidate_for_patient.copy().reset_index(drop=True)

# Required nutrient columns
required_cols = [
    "food_name",
    "category_code",
    "urt",
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]

# Keep only rows with required values
milp_df = milp_df.dropna(subset=[
    "category_code",
    "energy_kcal_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]).copy()

# Make sure numeric columns are numeric
numeric_cols = [
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

for col in numeric_cols:
    if col in milp_df.columns:
        milp_df[col] = pd.to_numeric(milp_df[col], errors="coerce").fillna(0)

print("Total MILP candidate foods:", len(milp_df))

display(
    milp_df.groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

Total MILP candidate foods: 181


,category_code,count
5,S,132
0,B,17
1,LH,10
4,MP,8
2,LN,7
3,M,5
6,SS,2


In [34]:
# ============================================================
# 2. Create MILP model
# ============================================================

model = pulp.LpProblem("DM_HT_Obesity_Menu_Recommendation", pulp.LpMinimize)

# Decision variable:
# x[i] = number of portions selected for food i
# Upper bound 2 means one food can be selected max 2 portions/day
x = {
    i: pulp.LpVariable(f"x_{i}", lowBound=0, upBound=2, cat="Integer")
    for i in milp_df.index
}

Formulation total of nutrion in daily menu based on combination from MILP

In [35]:
# ============================================================
# 3. Define nutrient totals
# ============================================================

total_energy = pulp.lpSum(
    x[i] * milp_df.loc[i, "energy_kcal_per_portion"]
    for i in milp_df.index
)

total_protein = pulp.lpSum(
    x[i] * milp_df.loc[i, "protein_g_per_portion"]
    for i in milp_df.index
)

total_fat = pulp.lpSum(
    x[i] * milp_df.loc[i, "fat_g_per_portion"]
    for i in milp_df.index
)

total_carb = pulp.lpSum(
    x[i] * milp_df.loc[i, "carb_g_per_portion"]
    for i in milp_df.index
)

total_sodium = pulp.lpSum(
    x[i] * milp_df.loc[i, "sodium_mg_per_portion"]
    for i in milp_df.index
)

total_fiber = pulp.lpSum(
    x[i] * milp_df.loc[i, "fiber_g_per_portion"]
    for i in milp_df.index
)

total_potassium = pulp.lpSum(
    x[i] * milp_df.loc[i, "potassium_mg_per_portion"]
    for i in milp_df.index
) if "potassium_mg_per_portion" in milp_df.columns else 0

total_calcium = pulp.lpSum(
    x[i] * milp_df.loc[i, "calcium_mg_per_portion"]
    for i in milp_df.index
) if "calcium_mg_per_portion" in milp_df.columns else 0

In [36]:
portion_plan = {
    1100: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 0, "M": 3, "G": 1},
    1200: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 1, "M": 3, "G": 2},
    1300: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 2, "SS": 1, "M": 4, "G": 2},
    1400: {"MP": 3, "LH": 3, "LN": 3, "S": 2, "B": 2, "SS": 0, "M": 3, "G": 3},
    1500: {"MP": 3, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 0, "M": 4, "G": 3},
    1600: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 2, "SS": 0, "M": 4, "G": 2},
    1700: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 3, "G": 2},
    1800: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    1900: {"MP": 4, "LH": 4, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    2000: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2100: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2200: {"MP": 4, "LH": 3, "LN": 3, "S": 4, "B": 4, "SS": 2, "M": 5, "G": 4},
    2300: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 0, "M": 5, "G": 4},
    2400: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
    2500: {"MP": 5.5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
}

In [37]:
def get_disease_constraints(diet_type, energy_kcal):
    """
    diet_type options:
    HT
    HT_OBESITY
    DM
    DM_OBESITY
    DM_HT
    DM_HT_OBESITY
    """

    constraints = {
        "energy_target": energy_kcal,
        "energy_mode": "maintenance",
        "carb_min_g": None,
        "carb_max_g": None,
        "fat_min_g": None,
        "fat_max_g": None,
        "sodium_max_mg": None,
        "potassium_min_mg": None,
        "calcium_min_mg": None,
        "fiber_min_g": None,
        "vegetable_fruit_min_portions": None,
        "sugar_max_portions": None
    }

    # 1. Hipertensi tanpa obesitas
    if diet_type == "HT":
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 2. Hipertensi dengan obesitas
    elif diet_type == "HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = energy_kcal * 0.55 / 4
        constraints["carb_max_g"] = energy_kcal * 0.65 / 4
        constraints["fat_min_g"] = energy_kcal * 0.20 / 9
        constraints["fat_max_g"] = energy_kcal * 0.25 / 9
        constraints["sodium_max_mg"] = 2300
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 3. DM tanpa obesitas
    elif diet_type == "DM":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sugar_max_portions"] = 0

    # 4. DM dengan obesitas
    elif diet_type == "DM_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 5. DM + Hipertensi tanpa obesitas
    elif diet_type == "DM_HT":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2300
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 6. DM + Hipertensi + Obesitas
    elif diet_type == "DM_HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = 130
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2000
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    return constraints

In [38]:
# ============================================================
# 3. Pick nearest available energy level
# ============================================================

def get_nearest_energy_level(energy_kcal, available_levels):
    return min(available_levels, key=lambda x: abs(x - energy_kcal))


def adjust_portion_plan_for_disease(base_plan, diet_type):
    plan = base_plan.copy()

    # DM: sugar should be avoided or minimized
    if "DM" in diet_type:
        plan["G"] = 0

    # Obesity: reduce sugar and oil portions
    if "OBESITY" in diet_type:
        plan["G"] = 0
        if "M" in plan:
            plan["M"] = max(0, plan["M"] - 1)

    # DM obesity / DM+HT: ensure vegetable + fruit >= 5 portions
    if diet_type in ["DM_OBESITY", "DM_HT", "DM_HT_OBESITY"]:
        current_sb = plan.get("S", 0) + plan.get("B", 0)
        if current_sb < 5:
            plan["S"] = plan.get("S", 0) + (5 - current_sb)

    return plan

In [39]:
# ============================================================
# 5. Example patient target
# ============================================================

diet_type = "DM_HT_OBESITY"
energy_target = 1600

constraints = get_disease_constraints(diet_type, energy_target)

nearest_energy = get_nearest_energy_level(
    energy_target,
    list(portion_plan.keys())
)

base_plan = portion_plan[nearest_energy]
adjusted_plan = adjust_portion_plan_for_disease(base_plan, diet_type)

candidate_for_patient = filter_foods_for_disease(food_candidates, diet_type)

print("Diet type:", diet_type)
print("Energy target:", energy_target)
print("Nearest portion plan:", nearest_energy)
print("\nDisease constraints:")
print(constraints)

print("\nBase portion plan:")
print(base_plan)

print("\nAdjusted portion plan:")
print(adjusted_plan)

print("\nTotal food candidates before filter:", len(food_candidates))
print("Total food candidates after disease filter:", len(candidate_for_patient))

category_after_filter = (
    candidate_for_patient
    .groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(category_after_filter)

Diet type: DM_HT_OBESITY
Energy target: 1600
Nearest portion plan: 1600

Disease constraints:
{'energy_target': 1600, 'energy_mode': 'deficit', 'carb_min_g': 130, 'carb_max_g': None, 'fat_min_g': None, 'fat_max_g': None, 'sodium_max_mg': 2000, 'potassium_min_mg': None, 'calcium_min_mg': None, 'fiber_min_g': 40.0, 'vegetable_fruit_min_portions': 5, 'sugar_max_portions': 0}

Base portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 4, 'G': 2}

Adjusted portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 3, 'G': 0}

Total food candidates before filter: 222
Total food candidates after disease filter: 211


,category_code,count
5,S,155
0,B,19
1,LH,13
2,LN,8
4,MP,8
3,M,5
6,SS,3


In [40]:
for category, required_portion in adjusted_plan.items():
    required_portion_int = int(required_portion)

    category_indices = milp_df[
        milp_df["category_code"] == category
    ].index.tolist()

    # If category has no food and required portion is 0, skip safely
    if len(category_indices) == 0 and required_portion_int == 0:
        continue

    # If category has no food but required portion > 0, this is a real problem
    if len(category_indices) == 0 and required_portion_int > 0:
        print(f"Warning: no candidate food for category {category}, but required portion is {required_portion_int}")
        continue

    model += (
        pulp.lpSum(x[i] for i in category_indices) == required_portion_int,
        f"portion_{category}"
    )

Ensure Constrain nutrion maxiumum around 10 percent

In [41]:
# ============================================================
# 5. Disease-specific nutrient constraints
# ============================================================

energy_target = constraints["energy_target"]
energy_min = energy_target * 0.90
energy_max = energy_target * 1.10

# Energy range
model += total_energy >= energy_min, "energy_min"
model += total_energy <= energy_max, "energy_max"

# Carbohydrate minimum
if constraints.get("carb_min_g") is not None:
    model += total_carb >= constraints["carb_min_g"], "carb_min"

# Carbohydrate maximum, if available
if constraints.get("carb_max_g") is not None:
    model += total_carb <= constraints["carb_max_g"], "carb_max"

# Fat minimum, if available
if constraints.get("fat_min_g") is not None:
    model += total_fat >= constraints["fat_min_g"], "fat_min"

# Fat maximum, if available
if constraints.get("fat_max_g") is not None:
    model += total_fat <= constraints["fat_max_g"], "fat_max"

# Sodium maximum
if constraints.get("sodium_max_mg") is not None:
    model += total_sodium <= constraints["sodium_max_mg"], "sodium_max"

# Fiber minimum
if constraints.get("fiber_min_g") is not None:
    model += total_fiber >= constraints["fiber_min_g"], "fiber_min"

In [42]:
# ============================================================
# 6. Objective function: minimize energy deviation
# ============================================================

energy_dev_pos = pulp.LpVariable("energy_dev_pos", lowBound=0)
energy_dev_neg = pulp.LpVariable("energy_dev_neg", lowBound=0)

model += total_energy - energy_target == energy_dev_pos - energy_dev_neg

# Objective:
# minimize energy deviation
# plus small penalty for sodium
model += (
    energy_dev_pos
    + energy_dev_neg
    + 0.01 * total_sodium
), "minimize_energy_deviation_and_sodium"

In [43]:
# ============================================================
# 7. Solve model
# ============================================================

solver = pulp.PULP_CBC_CMD(msg=True)
model.solve(solver)

print("Solver status:", pulp.LpStatus[model.status])
print("Objective value:", pulp.value(model.objective))

Solver status: Optimal
Objective value: 0.9624999999999999


In [44]:
# ============================================================
# 8. Extract selected foods
# ============================================================

selected_items = []

for i in milp_df.index:
    selected_portion = x[i].value()

    if selected_portion is not None and selected_portion > 0:
        row = milp_df.loc[i].copy()
        row["selected_portions"] = selected_portion
        selected_items.append(row)

milp_menu = pd.DataFrame(selected_items)

display(
    milp_menu[[
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "selected_portions",
        "energy_kcal_per_portion",
        "protein_g_per_portion",
        "fat_g_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]]
)

,food_name,category_code,urt,gram_per_portion,selected_portions,energy_kcal_per_portion,protein_g_per_portion,fat_g_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
0,"Tepung beras, mentah",MP,8 Sendok Makan,50.0,2.0,176.50,3.500,0.250,40.000,2.50,1.20
2,"Makaroni, mentah",MP,1½ Gelas,50.0,2.0,176.50,4.350,0.200,39.350,2.50,2.45
14,"Tahu, mentah",LN,2 Potong Sedang,100.0,1.0,80.00,10.900,4.700,0.800,2.00,0.10
15,"Tempe kedelai, mentah",LN,2 potong sedang,50.0,2.0,100.50,10.400,4.400,6.750,4.50,0.70
76,"Daun pangi,segar",S,1 gelas,100.0,1.0,104.00,3.900,0.600,17.900,4.00,8.30
136,"Nangka muda, segar",S,1 gelas,100.0,2.0,57.00,2.000,0.400,11.300,1.10,8.30
171,"Duku, segar",B,10-16 buah sedang,80.0,2.0,50.40,0.800,0.160,12.880,1.60,3.44
194,"Sapi, daging, gemuk, segar",LH,1 potong sedang,35.0,1.0,95.55,6.125,7.700,0.000,32.55,0.00
198,"Cumi-cumi, segar",LH,1 ekor kecil,45.0,2.0,33.75,7.245,0.315,0.045,16.65,0.00
207,Minyak kedelai,M,1 sendok teh,5.0,1.0,44.15,0.000,4.995,0.000,0.00,0.00


In [45]:
# ============================================================
# 8. Evaluate menu against constraints
# ============================================================

def evaluate_menu(daily_totals, constraints):
    evaluation = []

    energy_total = daily_totals.get("energy_kcal_per_portion", np.nan)
    carb_total = daily_totals.get("carb_g_per_portion", np.nan)
    protein_total = daily_totals.get("protein_g_per_portion", np.nan)
    fat_total = daily_totals.get("fat_g_per_portion", np.nan)
    sodium_total = daily_totals.get("sodium_mg_per_portion", np.nan)
    potassium_total = daily_totals.get("potassium_mg_per_portion", np.nan)
    calcium_total = daily_totals.get("calcium_mg_per_portion", np.nan)
    fiber_total = daily_totals.get("fiber_g_per_portion", np.nan)

    # Energy: allow ±10% for rule-based prototype
    energy_target = constraints["energy_target"]
    energy_min = energy_target * 0.90
    energy_max = energy_target * 1.10

    evaluation.append({
        "constraint": "energy",
        "value": energy_total,
        "target_or_limit": f"{energy_min:.1f} - {energy_max:.1f}",
        "status": "pass" if energy_min <= energy_total <= energy_max else "not_pass"
    })

    # Carbohydrate minimum
    if constraints.get("carb_min_g") is not None:
        evaluation.append({
            "constraint": "carb_min",
            "value": carb_total,
            "target_or_limit": f">= {constraints['carb_min_g']:.1f}",
            "status": "pass" if carb_total >= constraints["carb_min_g"] else "not_pass"
        })

    # Carbohydrate maximum
    if constraints.get("carb_max_g") is not None:
        evaluation.append({
            "constraint": "carb_max",
            "value": carb_total,
            "target_or_limit": f"<= {constraints['carb_max_g']:.1f}",
            "status": "pass" if carb_total <= constraints["carb_max_g"] else "not_pass"
        })

    # Fat minimum
    if constraints.get("fat_min_g") is not None:
        evaluation.append({
            "constraint": "fat_min",
            "value": fat_total,
            "target_or_limit": f">= {constraints['fat_min_g']:.1f}",
            "status": "pass" if fat_total >= constraints["fat_min_g"] else "not_pass"
        })

    # Fat maximum
    if constraints.get("fat_max_g") is not None:
        evaluation.append({
            "constraint": "fat_max",
            "value": fat_total,
            "target_or_limit": f"<= {constraints['fat_max_g']:.1f}",
            "status": "pass" if fat_total <= constraints["fat_max_g"] else "not_pass"
        })

    # Sodium maximum
    if constraints.get("sodium_max_mg") is not None:
        evaluation.append({
            "constraint": "sodium_max",
            "value": sodium_total,
            "target_or_limit": f"<= {constraints['sodium_max_mg']:.1f}",
            "status": "pass" if sodium_total <= constraints["sodium_max_mg"] else "not_pass"
        })

    # Fiber minimum
    if constraints.get("fiber_min_g") is not None:
        evaluation.append({
            "constraint": "fiber_min",
            "value": fiber_total,
            "target_or_limit": f">= {constraints['fiber_min_g']:.1f}",
            "status": "pass" if fiber_total >= constraints["fiber_min_g"] else "not_pass"
        })

    # Potassium minimum
    if constraints.get("potassium_min_mg") is not None:
        evaluation.append({
            "constraint": "potassium_min",
            "value": potassium_total,
            "target_or_limit": f">= {constraints['potassium_min_mg']:.1f}",
            "status": "pass" if potassium_total >= constraints["potassium_min_mg"] else "not_pass"
        })

    # Calcium minimum
    if constraints.get("calcium_min_mg") is not None:
        evaluation.append({
            "constraint": "calcium_min",
            "value": calcium_total,
            "target_or_limit": f">= {constraints['calcium_min_mg']:.1f}",
            "status": "pass" if calcium_total >= constraints["calcium_min_mg"] else "not_pass"
        })

    return pd.DataFrame(evaluation)




In [46]:
# ============================================================
# 9. Calculate MILP nutrient totals
# ============================================================

nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

existing_nutrient_cols = [
    col for col in nutrient_cols if col in milp_menu.columns
]

milp_totals = {}

for col in existing_nutrient_cols:
    milp_totals[col] = (
        milp_menu[col] * milp_menu["selected_portions"]
    ).sum()

milp_totals = pd.Series(milp_totals)

display(
    milp_totals
    .reset_index()
    .rename(columns={"index": "nutrient", 0: "total"})
)

,nutrient,total
0,energy_kcal_per_portion,1600.000
1,protein_g_per_portion,77.615
2,fat_g_per_portion,39.245
3,carb_g_per_portion,239.350
4,sodium_mg_per_portion,96.250
5,potassium_mg_per_portion,1800.145
6,calcium_mg_per_portion,810.100
7,fiber_g_per_portion,40.580


In [47]:
milp_evaluation = evaluate_menu(
    milp_totals,
    constraints
)

display(milp_evaluation)

,constraint,value,target_or_limit,status
0,energy,1600.00,1440.0 - 1760.0,pass
1,carb_min,239.35,>= 130.0,pass
2,sodium_max,96.25,<= 2000.0,pass
3,fiber_min,40.58,>= 40.0,pass


In [48]:
selected_category_count = (
    milp_menu
    .groupby("category_code")["selected_portions"]
    .sum()
    .reset_index(name="selected_portions")
)

display(selected_category_count)

print("Adjusted portion plan:")
print(adjusted_plan)

,category_code,selected_portions
0,B,2.0
1,LH,3.0
2,LN,3.0
3,M,3.0
4,MP,4.0
5,S,3.0


Adjusted portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 3, 'G': 0}


In [49]:
milp_menu.to_csv("milp_menu_dm_ht_obesity.csv", index=False)
milp_evaluation.to_csv("milp_menu_evaluation_dm_ht_obesity.csv", index=False)

milp_totals.reset_index().rename(
    columns={"index": "nutrient", 0: "total"}
).to_csv("milp_menu_nutrient_total_dm_ht_obesity.csv", index=False)

print("Saved:")
print("- milp_menu_dm_ht_obesity.csv")
print("- milp_menu_evaluation_dm_ht_obesity.csv")
print("- milp_menu_nutrient_total_dm_ht_obesity.csv")

Saved:
- milp_menu_dm_ht_obesity.csv
- milp_menu_evaluation_dm_ht_obesity.csv
- milp_menu_nutrient_total_dm_ht_obesity.csv
